# 05 · place2vec: places that "look like work"

Word2Vec on sequences of visits: each device-day is split into sessions (gap >
`corpus.session_gap_hours`), every visit becomes a token, and places used like an office
end up close to the `WORK` tokens in embedding space. Separate models per period are
rotated into a common space (orthogonal Procrustes on shared HOME / WORK anchors) so
similarities can be compared over time. Method: `notebook_docs/methodology/PLACE2VEC.md`;
parameters: `configs/place2vec.json` (paper) and `configs/place2vec_demo.json` (demo overrides).

Tokenisation schemes (`remote_work_utils.place2vec.tokenize`):

| scheme | home / work | visit at a POI | visit elsewhere |
|---|---|---|---|
| 0 | `HOME` / `WORK` | `CAT_<category>` | `CTX_<context>` |
| 1 | `HOME_<area>` | `POI_<id>` | `GH_<geohash>` |
| 2 | `HOME_<area>` | `POI_<id>` | `CENSUS_<area>` |
| 3 | `HOME_<area>` | `POI_<id>_WEEKDAY_<day part>` | `GH_<geohash>_<day part>` |
| 5 | `HOME_<core>_WEEKDAY_<part>` | `POI_<cat>_<core>_<loc>_WEEKDAY_<part>` | `GH_<core>_WEEKDAY_<part>` |

The paper uses scheme 3, so each POI gets one vector per day part.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from remote_work_utils import place2vec as p2v
from remote_work_utils.pipeline import output_path
from remote_work_utils.plotting import mean_by
from remote_work_utils.settings import load_settings, resolve_path

settings = load_settings("../configs/demo.json")
config = load_settings("../configs/place2vec_demo.json")
places = pd.read_csv(resolve_path(settings, "places"))
category = places.set_index("place_id")["category"]

visits = pd.concat(
    [pd.read_parquet(output_path(settings, "visits", m)).assign(month=m) for m in settings["aggregation"]["months"]],
    ignore_index=True,
)
visits = visits.assign(dwell_geohash=visits["geohash"], pois_category=visits["poi_id"].map(category))

## Sentences

In [ ]:
for scheme in [0, 1, 2, 3, 5]:
    sentences = p2v.build_sentences(visits.assign(poi_location_suits_rw=1), config, settings["calendar"], scheme=scheme)
    longest = max(sentences, key=len)
    print(f"scheme {scheme}: {len(sentences):,} sentences; e.g. {' → '.join(longest[:5])}")

## One model per period, aligned to the first

In [ ]:
models = {}
for period, period_months in config["periods"].items():
    sentences = p2v.build_sentences(visits[visits["month"].isin(period_months)], config, settings["calendar"])
    models[period] = p2v.train_word2vec(sentences, config["word2vec"])
    print(f"{period}: {len(sentences):,} sentences, vocabulary {len(models[period].wv)}")

reference = next(iter(models))
anchors = p2v.shared_home_work_anchors(list(models.values()))
for period in list(models)[1:]:
    models[period], _ = p2v.align_procrustes(models[reference], models[period], anchors, config["alignment"]["min_anchor"])
print(f"{len(anchors)} shared HOME/WORK anchors")

## Similarity of each POI to work, by day part
A POI qualifies when its maximum daytime similarity to the `WORK` centroid exceeds
`similarity.work_threshold` while its evening tokens stay below it.

In [ ]:
rows = []
for period, model in models.items():
    centroid, _ = p2v.build_work_centroid(model)
    similarities = p2v.poi_period_similarities(model, {"WORK": centroid}, config["similarity"]["poi_periods"])
    flags = p2v.flag_daytime_work_places(similarities, config["similarity"])
    rows.append(flags.assign(period=period))
flags = pd.concat(rows, ignore_index=True).merge(
    places.rename(columns={"place_id": "poi_id"})[["poi_id", "category", "place_group"]], on="poi_id"
)
pd.crosstab([flags["period"], flags["place_group"]], flags["qualifies_remote_work"])

In [ ]:
daytime = flags.dropna(subset=["max_daytime_work_sim"])
print("Places with daytime weekday tokens:", daytime.groupby("period")["place_group"].value_counts().to_dict())

summary = mean_by(daytime, ["period"], "max_daytime_work_sim").set_index("period").reindex(list(models))
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(range(len(summary)), summary["mean"],
            yerr=[summary["mean"] - summary["ci_low"], summary["ci_high"] - summary["mean"]],
            fmt="o", capsize=3, color="#2f855a", label="Work-friendly places (mean ± 95% CI)")
ax.axhline(config["similarity"]["work_threshold"], ls="--", color="grey", label="Work threshold")
ax.set_xticks(range(len(summary)))
ax.set_xticklabels([p.replace("_", " ") for p in summary.index])
ax.set_title("Daytime similarity of POIs to the WORK centroid")
ax.set_xlabel("Period")
ax.set_ylabel("Max daytime cosine similarity")
ax.legend()
plt.show()

> **Fig. 5.1.** Maximum daytime cosine similarity of POI tokens to the `WORK` centroid, by period
> (mean ± 95% CI over POIs). Only work-friendly places have weekday daytime tokens frequent
> enough to be embedded — leisure places are visited in the evening and at weekends — and all
> of them stay above the work threshold. *Notes:* scheme 3, Procrustes-aligned models; synthetic
> data, one month per period.

## Frequent sequences
N-grams, generalised to categories and the home area's city.

In [ ]:
stat_areas = gpd.read_file(resolve_path(settings, "stat_areas"))
area_names = stat_areas.set_index(settings["stat_areas"]["id_column"])[settings["stat_areas"]["name_column"]]
lookups = p2v.TokenLookups(area_type={str(k): v for k, v in area_names.items()}, poi_category=category.to_dict())
sentences = p2v.build_sentences(visits, config, settings["calendar"])
pd.DataFrame(p2v.top_ngrams(sentences, 3, 10, lookups=lookups, by_area_type=True, by_category=True),
             columns=["trigram", "share of trigrams (%)"])